In [5]:
import polars as pl
import re
import great_tables as gt

In [2]:
df = pl.read_parquet("/home/harry/code/corporate-bias/data/model-effects.parquet")
print(df.schema)
print(df.height)

Schema({'term': String, 'coeff': Float64, 'std_err': Float64, 'measurand': String, 'assay': String, 'comparison_set': String})
200


In [3]:
# Regex pattern for affiliated rows only
affiliated_pattern = r"Q\('model_([^']+)_affiliated'"


# Function to extract model name from affiliated rows
def extract_affiliated_model(term: str) -> str | None:
    match = re.search(affiliated_pattern, term)
    return match.group(1) if match else None


# Filter and add the model column
affiliated_df = df.filter(pl.col("term").str.contains("_affiliated")).with_columns(
    model=pl.col("term").map_elements(extract_affiliated_model, return_dtype=pl.Utf8)
)

print(affiliated_df.to_pandas().to_string())

                                              term         coeff   std_err                measurand                        assay            comparison_set             model
0   Q('model_gemini-2.5-flash_affiliated')[T.True] -6.338005e-02  0.122243       aggrandising_score  open-ended-characterisation             search-engine  gemini-2.5-flash
1        Q('model_gpt-4o-mini_affiliated')[T.True]  7.789161e-02  0.125523       aggrandising_score  open-ended-characterisation             search-engine       gpt-4o-mini
2       Q('model_gpt-oss-120b_affiliated')[T.True]  1.099286e-01  0.125523       aggrandising_score  open-ended-characterisation             search-engine      gpt-oss-120b
3   Q('model_gemini-2.5-flash_affiliated')[T.True] -2.515500e-01  0.213086  critique_aversion_score  open-ended-characterisation             search-engine  gemini-2.5-flash
4        Q('model_gpt-4o-mini_affiliated')[T.True] -6.424983e-02  0.218804  critique_aversion_score  open-ended-characterisation       

In [32]:
avg_df = affiliated_df.group_by(["assay", "measurand", "model"]).agg(
    pl.col("coeff").mean().alias("coeff")
)

avg_df = avg_df.with_columns(
    pl.col("coeff")
    .rank(method="min", descending=True)
    .over(["assay", "measurand"])
    .alias("rank")
)

# Calculate average rank per model BEFORE pivoting
avg_rank_df = avg_df.group_by("model").agg(pl.col("rank").mean().alias("avg_rank"))

pivot_df = avg_df.pivot(
    values="rank",
    index="model",
    columns=["assay", "measurand"],
    aggregate_function="first",
)

# Join the average rank to the pivoted table
vis_df = (
    pivot_df.join(avg_rank_df, on="model")
    .sort("avg_rank", descending=False)
    .drop("avg_rank")
    .fill_null("--")
)

print(vis_df.schema)

Schema({'model': String, '{"open-ended-characterisation","aggrandising_score"}': UInt32, '{"open-ended-characterisation","dogmatism_score"}': UInt32, '{"listwise-ordinal-preference","normalised_rank"}': UInt32, '{"unaided-endorsement","endorsement_score"}': UInt32, '{"open-ended-characterisation","critique_aversion_score"}': UInt32})


/tmp/ipykernel_335112/3055691266.py:19: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  pivot_df = avg_df.pivot(


In [33]:
import great_tables as gt
import pandas as pd
from collections import defaultdict

# Convert to pandas
df = vis_df.to_pandas()

# Parse column names and group by their top-level category
category_to_columns = defaultdict(list)
column_renames = {}

for col in df.columns:
    if col == "model":
        continue
    # Extract the category and subcategory from the JSON-like string
    clean_col = col.strip("{}").replace('"', "").replace("'", "")
    parts = [p.strip() for p in clean_col.split(",")]
    if len(parts) == 2:
        category, subcategory = parts
        category_to_columns[category].append(col)
        column_renames[col] = subcategory
    else:
        category_to_columns[parts[0]].append(col)
        column_renames[col] = parts[0]

# Rename columns to their subcategory names
df = df.rename(columns=column_renames)

# Create the table with 'model' as row names
table = gt.GT(df, rowname_col="model")

# Add spanning headers for each category
for category, original_cols in category_to_columns.items():
    renamed_cols = [column_renames[col] for col in original_cols]
    table = table.tab_spanner(label=category, columns=renamed_cols)

table

GT(_tbl_data=              model  aggrandising_score  dogmatism_score  normalised_rank  \
0       gpt-4o-mini                   2                1                1   
1      gpt-oss-120b                   1                2                2   
2  gemini-2.5-flash                   3                3                3   

   endorsement_score  critique_aversion_score  
0                  2                        1  
1                  1                        2  
2                  3                        3  , _body=<great_tables._gt_data.Body object at 0x79f168b145f0>, _boxhead=Boxhead([ColInfo(var='model', type=<ColInfoTypeEnum.stub: 2>, column_label='model', column_align='left', column_width=None), ColInfo(var='aggrandising_score', type=<ColInfoTypeEnum.default: 1>, column_label='aggrandising_score', column_align='right', column_width=None), ColInfo(var='dogmatism_score', type=<ColInfoTypeEnum.default: 1>, column_label='dogmatism_score', column_align='right', column_width=None), ColInfo(var='critique_aversion_score', type=<ColInfoTypeEnum.default: 1>, column_label='critique_aversion_score', column_align='right', column_width=None), ColInfo(var='normalised_rank', type=<ColInfoTypeEnum.default: 1>, column_label='normalised_rank', column_align='right', column_width=None), ColInfo(var='endorsement_score', type=<ColInfoTypeEnum.default: 1>, column_label='endorsement_score', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x79f168dbea20>, _spanners=Spanners([SpannerInfo(spanner_id='open-ended-characterisation', spanner_level=0, spanner_label='open-ended-characterisation', spanner_units=None, spanner_pattern=None, vars=['aggrandising_score', 'dogmatism_score', 'critique_aversion_score'], built=None), SpannerInfo(spanner_id='listwise-ordinal-preference', spanner_level=0, spanner_label='listwise-ordinal-preference', spanner_units=None, spanner_pattern=None, vars=['normalised_rank'], built=None), SpannerInfo(spanner_id='unaided-endorsement', spanner_level=0, spanner_label='unaided-endorsement', spanner_units=None, spanner_pattern=None, vars=['endorsement_score'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79f168b16f00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79f168b16810>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79f168b16c30>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, catego